# 🏔️ Topic 07: Cloud Data Lakes, Parquet & Delta Lake

## 1. Storage Formats Comparison

| Format | Structure | Compression | Read Speed | Predicate Pushdown |
| :--- | :--- | :--- | :--- | :--- |
| **CSV** | Row-based | Poor | Slow | ❌ No |
| **JSON** | Row-based | Moderate | Slow | ❌ No |
| **Parquet** | Columnar | High (Snappy/Gzip) | Fast | ✅ Yes |
| **Delta Lake** | Columnar + ACID Log | High | Ultra Fast | ✅ Yes (Z-Ordering) |

---

## 2. Delta Lake Features
- **ACID Transactions:** Prevents corrupt data writes during cluster crashes.
- **Time Travel:** Query historic data versions (`VERSION AS OF` or `TIMESTAMP AS OF`).
- **Schema Enforcement & Evolution:** Blocks malformed columns automatically.

---

## 3. Hands-on: Partitioned Parquet Storage & Metadata Inspection


In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import os
import shutil

spark = SparkSession.builder.master("local[*]").appName("DataLake_Demo").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

output_parquet_dir = "partitioned_sales_data"
if os.path.exists(output_parquet_dir):
    shutil.rmtree(output_parquet_dir)

# Create synthetic sales dataset across multiple regions and years
data = [
    ("2026", "US-East", "Laptops", 1500.0),
    ("2026", "US-East", "Mice", 25.0),
    ("2026", "EU-West", "Laptops", 1600.0),
    ("2025", "US-East", "Monitors", 300.0),
    ("2025", "EU-West", "Mice", 30.0),
]

df_sales = spark.createDataFrame(data, ["year", "region", "product", "revenue"])

# Write partitioned Parquet files by year and region
print("💾 Writing Partitioned Parquet Files to disk...")
df_sales.write \
    .partitionBy("year", "region") \
    .mode("overwrite") \
    .parquet(output_parquet_dir)

print("\n📂 Folder Structure Generated on Disk:")
for root, dirs, files in os.walk(output_parquet_dir):
    level = root.replace(output_parquet_dir, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        if f.endswith(".parquet"):
            print(f"{subindent}{f[:25]}...")

# Read back with Predicate Pushdown (only reads required partition directory)
print("\n⚡ Reading Back Partition 'year=2026/region=US-East':")
df_filtered = spark.read.parquet(output_parquet_dir).filter((F.col("year") == "2026") & (F.col("region") == "US-East"))
df_filtered.show()
